# LAB: Retrieval-Augmented Generation (RAG)
## Génération Augmentée par Récupération
### Master SDIA - Prof. RETAL SARA

Ce notebook couvre la création de systèmes RAG pour extraire et augmenter les réponses des LLMs avec des données externes.

## ⚙️ SETUP - Installation et Configuration

### 1️⃣ Installation des dépendances

Exécutez cette commande dans votre terminal:

```bash
uv add langchain langchain-ollama langchain-community langchain-text-splitters \
    langchain-huggingface sentence-transformers pypdf python-dotenv pydantic langgraph
```

### 2️⃣ Vérification des installations

In [ ]:
import sys
import socket

print("\n" + "="*70)
print("VÉRIFICATION COMPLÈTE DU SETUP RAG")
print("="*70 + "\n")

print(f"✅ Python version: {sys.version}")
print()

# Vérifier les packages importants
packages_to_check = [
    'langchain',
    'langchain_community',
    'langchain_text_splitters',
    'pypdf',
    'sentence_transformers',
    'dotenv',
    'langgraph'
]

for package in packages_to_check:
    try:
        mod = __import__(package.replace('_', '-'))
        print(f"✅ {package}: installé")
    except ImportError:
        print(f"❌ {package}: NON INSTALLÉ")

print()

# Vérifier Ollama
print("Vérification de Ollama:")
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(2)
result = sock.connect_ex(('127.0.0.1', 11434))
sock.close()

if result == 0:
    print("✅ Ollama est accessible sur http://localhost:11434")
else:
    print("❌ Ollama n'est pas accessible")
    print("   Conseil: Exécutez 'ollama serve' dans un terminal")

print("\n" + "="*70)

### 3️⃣ Téléchargement des fichiers exemple

Pour ce lab, vous avez besoin de:
- **Fichier PDF**: Utilisez votre propre PDF ou créez-en un
- **Base de données SQLite**: Téléchargez `Chinook.db` depuis:
  https://github.com/lerocha/chinook-database/blob/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite

Placez ces fichiers dans le même dossier que ce notebook.

---

## 📚 PARTIE 1: RAG avec PDF

### Qu'est-ce que le RAG?

**RAG (Retrieval-Augmented Generation)** = Récupération + Génération

1. **Récupération**: Chercher les informations pertinentes dans vos données
2. **Augmentation**: Ajouter ces informations au contexte du LLM
3. **Génération**: Le LLM génère une réponse basée sur les données récupérées

**Avantages**:
- ✅ Réponses basées sur vos données
- ✅ Pas besoin de fine-tuning du modèle
- ✅ Facile à mettre à jour
- ✅ Réduit les hallucinations

### Étape 1: Chargement d'un fichier PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

# Vérifier si un fichier PDF existe
pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]

if pdf_files:
    pdf_file = pdf_files[0]  # Utiliser le premier PDF trouvé
    print(f"📄 PDF trouvé: {pdf_file}")
else:
    pdf_file = "acmecorp-employee-handbook.pdf"  # Exemple
    print(f"⚠️  Aucun PDF trouvé. Utilisation d'un exemple: {pdf_file}")
    print("   Veuillez placer votre PDF dans le dossier courant.")

try:
    # Charger le fichier PDF
    loader = PyPDFLoader(pdf_file)
    data = loader.load()
    
    print(f"\n✅ PDF chargé avec succès!")
    print(f"   - Nombre de pages: {len(data)}")
    print(f"   - Premier extrait: {data[0].page_content[:200]}...")
except FileNotFoundError:
    print(f"❌ Fichier '{pdf_file}' non trouvé")
    print("   Créez un PDF exemple ou téléchargez-en un")
    data = None
except Exception as e:
    print(f"❌ Erreur: {e}")
    data = None

### Étape 2: Segmentation du texte en chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

if data:
    # Créer un splitter de texte
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # Taille de chaque chunk
        chunk_overlap=200,    # Chevauchement entre chunks
        add_start_index=True  # Ajouter l'index de départ
    )
    
    # Diviser les documents en chunks
    all_splits = text_splitter.split_documents(data)
    
    print(f"✅ Texte segmenté avec succès!")
    print(f"   - Nombre de chunks: {len(all_splits)}")
    print(f"   - Taille moyenne par chunk: {len(all_splits[0].page_content) if all_splits else 0} caractères")
    print(f"\nPremier chunk:")
    print(f"   {all_splits[0].page_content[:300]}...")
else:
    print("⚠️  Impossible de segmenter: aucun PDF chargé")
    all_splits = None

### Étape 3: Génération des embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print("⏳ Chargement du modèle d'embeddings (première utilisation: ~2-3 minutes)...\n")

try:
    # Utiliser un modèle HuggingFace pour les embeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    print("✅ Modèle d'embeddings chargé!")
    print(f"   - Modèle: sentence-transformers/all-MiniLM-L6-v2")
    print(f"   - Taille: 22.7 MB")
    print(f"   - Dimension: 384")
    
    # Tester avec un exemple
    test_embedding = embeddings.embed_query("Hello world")
    print(f"\n   - Test embedding: {len(test_embedding)} dimensions")
    print(f"   - Premiers valeurs: {test_embedding[:5]}")
except Exception as e:
    print(f"❌ Erreur: {e}")
    embeddings = None

### Étape 4: Création d'une base vectorielle en mémoire

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

if embeddings and all_splits:
    # Créer la base vectorielle en mémoire
    vector_store = InMemoryVectorStore(embeddings)
    
    print("✅ Base vectorielle créée en mémoire")
    print(f"   Type: InMemoryVectorStore")
    print(f"   État: Vide (en attente d'indexation)")
else:
    print("⚠️  Impossible de créer la base: embeddings ou documents manquants")
    vector_store = None

### Étape 5: Indexation des documents

In [ ]:
if vector_store and all_splits:
    print("⏳ Indexation des documents (peut prendre quelques secondes)...\n")
    
    try:
        # Ajouter les documents à la base vectorielle
        ids = vector_store.add_documents(documents=all_splits)
        
        print(f"✅ Documents indexés avec succès!")
        print(f"   - Nombre de documents indexés: {len(ids)}")
        print(f"   - Premiers IDs: {ids[:3]}")
    except Exception as e:
        print(f"❌ Erreur lors de l'indexation: {e}")
else:
    print("⚠️  Impossible d'indexer: base vectorielle ou documents manquants")

### Étape 6: Recherche sémantique

In [ ]:
if vector_store:
    # Exemple de requête de recherche
    query = "How many days of vacation does an employee get in their first year?"
    
    print(f"🔍 Recherche: '{query}'\n")
    
    try:
        # Effectuer une recherche sémantique
        results = vector_store.similarity_search(query, k=3)  # Top 3 résultats
        
        print(f"✅ {len(results)} résultat(s) trouvé(s):\n")
        
        for i, result in enumerate(results, 1):
            print(f"Résultat {i}:")
            print(f"  Source: {result.metadata if result.metadata else 'Unknown'}")
            print(f"  Contenu: {result.page_content[:300]}...")
            print()
    except Exception as e:
        print(f"❌ Erreur lors de la recherche: {e}")
else:
    print("⚠️  Impossible de rechercher: base vectorielle non disponible")

### Étape 7: Création d'un Agent RAG

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

if vector_store:
    # Créer un tool de recherche
    @tool
    def search_documents(query: str) -> str:
        """Recherche des informations pertinentes dans les documents.
        
        Args:
            query: La question ou requête de recherche
        """
        try:
            results = vector_store.similarity_search(query, k=1)
            if results:
                return results[0].page_content
            return "Aucun document pertinent trouvé."
        except Exception as e:
            return f"Erreur lors de la recherche: {e}"
    
    print("✅ Tool de recherche créé")
else:
    print("⚠️  Impossible de créer le tool: base vectorielle non disponible")
    search_documents = None

In [ ]:
if search_documents:
    try:
        # Initialiser le modèle Ollama
        model = ChatOllama(
            model="llama3.2:3b",
            temperature=0
        )
        
        # Créer l'agent RAG
        agent = create_agent(
            model=model,
            tools=[search_documents],
            system_prompt="You are a helpful assistant that searches the documents to answer questions. Always search for relevant information before answering."
        )
        
        print("✅ Agent RAG créé avec succès!")
        print(f"   - Modèle: llama3.2:3b")
        print(f"   - Tools: search_documents")
    except Exception as e:
        print(f"❌ Erreur lors de la création de l'agent: {e}")
        agent = None
else:
    print("⚠️  Impossible de créer l'agent: search_documents non disponible")
    agent = None

In [ ]:
if agent:
    # Poser une question à l'agent RAG
    question = HumanMessage(content="How many days of vacation does an employee get in their first year?")
    
    print(f"❓ Question: {question.content}\n")
    print("⏳ Recherche en cours...\n")
    
    try:
        response = agent.invoke(
            {"messages": [question]}
        )
        
        print("✅ Réponse de l'agent RAG:")
        print(response['messages'][-1].content)
    except ConnectionError as e:
        print("❌ ERREUR: Impossible de se connecter à Ollama")
        print("   Assurez-vous qu'Ollama est en cours d'exécution: 'ollama serve'")
    except Exception as e:
        print(f"❌ Erreur: {e}")
else:
    print("⚠️  Impossible d'invoquer l'agent: agent non disponible")

---

## 💾 PARTIE 2: RAG avec Base de Données SQL

### Qu'est-ce qu'un SQL Agent?

Un agent SQL utilise un LLM pour:
1. Comprendre les questions en langage naturel
2. Générer les requêtes SQL appropriées
3. Exécuter les requêtes
4. Formater et retourner les résultats

**Avantages**:
- ✅ Requêtes dynamiques sans coder SQL
- ✅ Accès aux données actuelles
- ✅ Données structurées et fiables

### Étape 1: Connexion à une base de données SQLite

In [ ]:
from langchain_community.utilities import SQLDatabase
import os

# Vérifier si la base Chinook existe
db_file = "Chinook.db"

if os.path.exists(db_file):
    print(f"📁 Base de données trouvée: {db_file}")
else:
    print(f"⚠️  Base de données '{db_file}' non trouvée")
    print("\nVous pouvez:")
    print("1. Télécharger Chinook.db depuis:")
    print("   https://github.com/lerocha/chinook-database")
    print("2. Utiliser votre propre base SQLite")
    print("3. Créer une base de démonstration")

try:
    # Connexion à la base de données
    db = SQLDatabase.from_uri(f"sqlite:///{db_file}")
    
    print(f"\n✅ Connexion établie à {db_file}")
    
    # Afficher le schéma
    print(f"\nSchéma de la base de données:")
    print(db.get_table_names())
except Exception as e:
    print(f"❌ Erreur de connexion: {e}")
    db = None

### Étape 2: Création d'un tool personnalisé pour requêtes SQL

In [ ]:
from langchain.tools import tool

if db:
    # Créer un tool pour exécuter des requêtes SQL
    @tool
    def sql_query(query: str) -> str:
        """Exécute une requête SQL et retourne les résultats.
        
        Args:
            query: La requête SQL à exécuter
        """
        try:
            print(f"🔄 Exécution: {query[:100]}...")
            result = db.run(query)
            return result
        except Exception as e:
            return f"Erreur SQL: {e}"
    
    print("✅ Tool SQL créé")
    
    # Tester le tool
    print("\n🧪 Test du tool SQL:")
    test_result = sql_query.invoke("SELECT * FROM Artist LIMIT 3")
    print(f"Résultat: {test_result}")
else:
    print("⚠️  Impossible de créer le tool: base de données non disponible")
    sql_query = None

### Étape 3: Création d'un agent SQL LLM

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

if sql_query and db:
    # Obtenir le schéma de la base
    schema_info = db.get_table_info()
    
    # Créer un system prompt détaillé
    system_prompt = f"""You are a SQL expert assistant. Your job is to:
1. Understand natural language questions
2. Generate appropriate SQL queries
3. Execute the query and return results in a human-readable format

Rules:
- Only use the sql_query tool
- Always write valid SQL
- Only use columns that exist in the schema
- If the information doesn't exist, say so clearly
- Format results nicely for users

Database Schema:
{schema_info}
"""
    
    try:
        # Initialiser le modèle Ollama
        model = ChatOllama(
            model="llama3.2:3b",
            temperature=0
        )
        
        # Créer l'agent SQL
        sql_agent = create_agent(
            model=model,
            tools=[sql_query],
            system_prompt=system_prompt
        )
        
        print("✅ Agent SQL créé avec succès!")
        print(f"   - Modèle: llama3.2:3b")
        print(f"   - Outil: sql_query")
        print(f"   - Tables disponibles: {', '.join(db.get_table_names())}")
    except Exception as e:
        print(f"❌ Erreur: {e}")
        sql_agent = None
else:
    print("⚠️  Impossible de créer l'agent: sql_query non disponible")
    sql_agent = None

### Étape 4: Interrogation via langage naturel

In [ ]:
if sql_agent:
    # Exemple de question en langage naturel
    question = HumanMessage(content="Give me the first 5 artists in the database")
    
    print(f"❓ Question: {question.content}\n")
    print("⏳ Génération de la requête SQL...\n")
    
    try:
        response = sql_agent.invoke(
            {"messages": [question]}
        )
        
        print("✅ Réponse de l'agent SQL:")
        print(response['messages'][-1].content)
    except ConnectionError as e:
        print("❌ ERREUR: Impossible de se connecter à Ollama")
        print("   Assurez-vous qu'Ollama est en cours d'exécution: 'ollama serve'")
    except Exception as e:
        print(f"❌ Erreur: {e}")
else:
    print("⚠️  Impossible d'interroger: agent non disponible")

### 🎯 Essayer d'autres requêtes

In [ ]:
if sql_agent:
    # Liste de requêtes à tester
    test_questions = [
        "How many customers do we have?",
        "What are the top 3 most popular albums by number of tracks?",
        "Show me customers from Canada",
    ]
    
    print("🧪 Tests d'autres requêtes:\n")
    
    for i, q in enumerate(test_questions, 1):
        print(f"\n{i}. Question: {q}")
        print("-" * 60)
        try:
            msg = HumanMessage(content=q)
            resp = sql_agent.invoke({"messages": [msg]})
            print(f"   {resp['messages'][-1].content[:200]}...")
        except Exception as e:
            print(f"   ❌ Erreur: {str(e)[:100]}")

---

## 📊 Comparaison: PDF RAG vs SQL Agent

| Aspect | PDF RAG | SQL Agent |
|--------|---------|----------|
| **Source de données** | Documents non-structurés (texte) | Données structurées (tables) |
| **Recherche** | Sémantique (embeddings) | Basée sur requêtes (SQL) |
| **Exactitude** | Approximative | Exacte |
| **Contexte** | Flexible | Limité au schéma |
| **Mise à jour** | Besoin de réindexation | Automatique |
| **Cas d'usage** | Manuels, articles, emails | Données métier, analytics |

### Quand utiliser quoi?

**Utilisez PDF RAG pour:**
- Manuels et documentation
- Articles et blogs
- Emails et messages
- Contenu non-structuré

**Utilisez SQL Agent pour:**
- Requêtes de données métier
- Analyses et rapports
- Données en temps réel
- Données transactionnelles

---

## 🚀 Cas d'usage avancés

### 1. Combiner PDF RAG + SQL Agent

```python
# Un agent avec accès aux deux sources
@tool
def search_docs(query: str) -> str:
    return vector_store.similarity_search(query)[0].page_content

@tool
def query_db(sql: str) -> str:
    return db.run(sql)

# Agent avec les deux tools
agent = create_agent(
    model=model,
    tools=[search_docs, query_db],
    system_prompt="You can search documents and query the database..."
)
```

### 2. Persistance de la base vectorielle

```python
from langchain_community.vectorstores import FAISS

# Sauvegarder
vector_store.save_local("my_vector_store")

# Charger
vector_store = FAISS.load_local("my_vector_store", embeddings)
```

### 3. Améliorer les résultats avec métadonnées

```python
from langchain_core.documents import Document

doc = Document(
    page_content="...",
    metadata={"source": "file.pdf", "page": 1, "author": "John"}
)
```

---

## 📝 Résumé des concepts clés

### RAG Pipeline:
1. **Chargement** → `PyPDFLoader`
2. **Segmentation** → `RecursiveCharacterTextSplitter`
3. **Embeddings** → `HuggingFaceEmbeddings`
4. **Indexation** → `InMemoryVectorStore`
5. **Recherche** → `similarity_search()`
6. **Génération** → Agent LLM

### SQL Agent Pipeline:
1. **Connexion** → `SQLDatabase.from_uri()`
2. **Tool création** → `@tool` decorator
3. **Agent création** → `create_agent()`
4. **Exécution** → LLM génère SQL
5. **Résultat** → Formattage naturel